# 07 - Window-Function Based Ranking
## Objective 8

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

Three window patterns: a partitioned ranking, a global ranking, and a rolling ordered frame. The
last one is the reason this objective exists. A trailing average per product cannot be expressed
with `groupBy` at all, because it needs a frame that moves with each row in time order.

In [ ]:
import os, sys, glob

_candidates = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
for _p in _candidates:
    if os.path.exists(os.path.join(_p, "da_common.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
else:
    raise FileNotFoundError("da_common.py not found. See KAGGLE_SETUP.md for the two setup options.")

from da_common import *

banner("Notebook 07 - Objective 8")
spark = get_spark("07 window functions")
df = load_analytical(spark).cache()
n_clean = df.count()
print(f"analytical dataset: {n_clean:,} rows")

### 1. dense_rank within a category partition

In [ ]:
products = (df.groupBy("parent_asin", "Category")
            .agg(F.count("*").alias("reviews"),
                 F.round(F.avg("rating"), 3).alias("avg_rating"),
                 F.sum("helpful_vote").alias("total_helpful")))
eligible = products.filter(F.col("reviews") >= MIN_REVIEWS).cache()

w_cat = Window.partitionBy("Category").orderBy(F.desc("avg_rating"), F.desc("reviews"))
ranked_products = (eligible.withColumn("rank_in_category", F.dense_rank().over(w_cat))
                   .filter(F.col("rank_in_category") <= 3)
                   .orderBy("Category", "rank_in_category"))
print("Top 3 products per category")
ranked_products.select("Category", "rank_in_category", "parent_asin",
                       "avg_rating", "reviews").show(truncate=False)
save_table(ranked_products, "obj8_top_products_per_category");

### 2. row_number over a global reviewer window

In [ ]:
reviewer = (df.groupBy("user_id")
            .agg(F.count("*").alias("reviews"),
                 F.round(F.avg("rating"), 2).alias("avg_rating"),
                 F.sum("helpful_vote").alias("total_helpful")))

w_rev = Window.orderBy(F.desc("reviews"), F.desc("total_helpful"))
ranked_reviewers = (reviewer.withColumn("activity_rank", F.row_number().over(w_rev))
                    .filter(F.col("activity_rank") <= 10)
                    .select("activity_rank", "user_id", "reviews", "avg_rating", "total_helpful"))
ranked_reviewers.show(truncate=False)
save_table(ranked_reviewers, "obj8_ranked_reviewers");

### 3. Rolling frame with rowsBetween

In [ ]:
w_roll = (Window.partitionBy("parent_asin").orderBy("timestamp")
          .rowsBetween(-(ROLLING_WINDOW - 1), Window.currentRow))
w_seq = Window.partitionBy("parent_asin").orderBy("timestamp")

rolling = (df.withColumn("rolling_avg_rating", F.round(F.avg("rating").over(w_roll), 3))
             .withColumn("review_seq", F.row_number().over(w_seq)))

busy = df.groupBy("parent_asin").count().orderBy(F.desc("count")).first()["parent_asin"]
example = (rolling.filter(F.col("parent_asin") == busy)
           .select("review_seq", "event_time", "rating", "rolling_avg_rating")
           .orderBy("review_seq"))
print(f"Trailing-{ROLLING_WINDOW} rolling average for the busiest product ({busy})")
example.show(10, truncate=False)

ex_pdf = example.toPandas()
save_table(ex_pdf.head(500), "obj8_rolling_average_example");

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.6))
ax.plot(ex_pdf["review_seq"], ex_pdf["rating"], alpha=0.25, color="#94a3b8",
        marker=".", linestyle="none", label="Individual ratings")
ax.plot(ex_pdf["review_seq"], ex_pdf["rolling_avg_rating"], color="#dc2626", lw=2,
        label=f"Trailing-{ROLLING_WINDOW} rolling average")
ax.set_title(f"Objective 8 - rolling average rating for product {busy}")
ax.set_xlabel("Review sequence (chronological)"); ax.set_ylabel("Rating")
ax.set_ylim(0.8, 5.2); ax.legend()
plt.tight_layout()
savefig(fig, "fig11_rolling_average", "Objective 8 - rolling window rating")
plt.show()

### Findings

The rolling series shows something a per-product average cannot: whether a product's reception is
stable or drifting. A flat rolling line and a declining one can produce the same lifetime average,
and only the second is a warning worth acting on. That is the practical argument for window
functions in this pipeline rather than a purely technical one.


In [ ]:
print(f"Tables in {TBL_DIR}")
for name in sorted(RESULTS):
    print(f"  {name:<40} {len(RESULTS[name]):>6,} rows")
print(f"\nFigures in {FIG_DIR}")
for name in sorted(FIGURES):
    print(f"  {name}.png")

In [ ]:
spark.stop()
print("Spark session stopped.")